# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
from lib.vector_db import VectorStoreManager
from lib.documents import Document
from dotenv import load_dotenv

E0000 00:00:1780290698.043618 7391429 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1780290698.044391 7391429 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1780290698.044396 7391429 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1780290698.044398 7391429 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pressure' registered more than once. Ignoring later registration.
E0000 00:00:1780290698.044400 7391429 instrument.cc:563] Metric with name 'grpc.resource_quota.memory_pressure_control_value' registered more than once. Ignoring later registration.


In [3]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CHROMA_OPENAI_API_KEY = os.getenv("CHROMA_OPENAI_API_KEY") or OPENAI_API_KEY

assert OPENAI_API_KEY is not None, "OPENAI_API_KEY missing in .env"
assert CHROMA_OPENAI_API_KEY is not None, "CHROMA_OPENAI_API_KEY missing in .env"

In [4]:
# Choose reusable vector-store manager
manager = VectorStoreManager(
    openai_api_key=CHROMA_OPENAI_API_KEY,
    persist_path="chromadb",
    embedding_model="text-embedding-3-small",
    openai_base_url="https://openai.vocareum.com/v1"
)

### Collection

In [5]:
store = manager.get_or_create_store("udaplay")

### Add documents

In [6]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

existing_ids = set(store.get()["ids"])
added, skipped = 0, 0
for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    if doc_id not in existing_ids:
        store.add(Document(id=doc_id, content=content, metadata=game))
        added += 1
    else:
        skipped += 1

print(f"Added {added} new documents, skipped {skipped} already-present documents.")

Added 15 new documents, skipped 0 already-present documents.


In [7]:
def show_results(query, n_results=3):
    results = store.query(query_texts=[query], n_results=n_results)
    print(f"Query: {query!r}")
    for meta, dist in zip(
        results["metadatas"][0],
        results["distances"][0]
    ):
        print(f"  -> [{meta['Platform']}] {meta['Name']} ({meta['YearOfRelease']})  dist={dist:.4f}")
    print()  # ==

In [8]:
show_results("realistic racing simulator on PlayStation")
show_results("open-world action adventure by Rockstar")
show_results("first 3D platformer Mario game")

Query: 'realistic racing simulator on PlayStation'
  -> [PlayStation 1] Gran Turismo (1997)  dist=0.3007
  -> [PlayStation 3] Gran Turismo 5 (2010)  dist=0.3374
  -> [PlayStation 2] Grand Theft Auto: San Andreas (2004)  dist=0.5942

Query: 'open-world action adventure by Rockstar'
  -> [PlayStation 2] Grand Theft Auto: San Andreas (2004)  dist=0.4718
  -> [Xbox One] Minecraft (2014)  dist=0.5923
  -> [PlayStation 4] Marvel's Spider-Man (2018)  dist=0.5986

Query: 'first 3D platformer Mario game'
  -> [Nintendo 64] Super Mario 64 (1996)  dist=0.3851
  -> [Super Nintendo Entertainment System (SNES)] Super Mario World (1990)  dist=0.4413
  -> [GameCube] Super Smash Bros. Melee (2001)  dist=0.5972

